In [1]:
from sae_lens import SAE 
from transformers import AutoModelForCausalLM, AutoTokenizer

import torch 
import numpy as np 

/Users/shayansadeghieh/Desktop/projects/mech_interp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Determine device 

In [2]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS")
else:
    device = torch.device("cpu")
    print("MPS device not found. Using CPU instead.")

Using MPS


# Load model onto GPU and tokenizer onto CPU

In [3]:
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token


In [4]:
def tokenize(input_text: str) -> torch.Tensor:    
    return gpt2_tokenizer(input_text, return_tensors="pt", padding=True).input_ids

def tokenize_and_embed(input_text: str) -> torch.Tensor:
    # Our model lives on the GPU, so we always put our tokenized tensor onto the gpu 
    return model.transformer.wte(tokenize(input_text).to(device))

def perform_inference(input_text: str) -> str:        
    input_ids = tokenize(input_text).to(device) # Our tokenizer output should live on the GPU 
    # input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(device)
    gen_tokens = model.generate(
        input_ids,
        do_sample=False, 
        temperature=0,
        max_length=100,
    )
    gen_text = gpt2_tokenizer.batch_decode(gen_tokens)[0]
    return gen_text

In [5]:
perform_inference("GPT2 is a model developed by OpenAI.")

/Users/shayansadeghieh/Desktop/projects/mech_interp/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


'GPT2 is a model developed by OpenAI. It is a model that is based on the concept of a "supercomputer" that can perform computations on a large number of data sets. It is a model that is based on the concept of a "supercomputer" that can perform computations on a large number of data sets. It is a model that is based on the concept of a "supercomputer" that can perform computations on a large number of data sets. It is'

# Load a sparse autoencoder 

In [10]:
sae, cfg_dict, sparsity = SAE.from_pretrained(
    release="gpt2-small-res-jb",  # see other options in sae_lens/pretrained_saes.yaml
    sae_id="blocks.1.hook_resid_pre",  # won't always be a hook point
    device='cpu',
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/Users/shayansadeghieh/Desktop/projects/mech_interp/.venv/lib/python3.12/site-packages/sae_lens/sae.py:145: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


In [11]:
params_dict = {}
for name, param in sae.named_parameters():
    params_dict[name] = param.data
    print(f"SAE has layer {name} with {param.size()} parameters")    

SAE has layer b_enc with torch.Size([24576]) parameters
SAE has layer W_dec with torch.Size([24576, 768]) parameters
SAE has layer W_enc with torch.Size([768, 24576]) parameters
SAE has layer b_dec with torch.Size([768]) parameters


In [12]:
model.transformer.h[11]

GPT2Block(
  (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (attn): GPT2SdpaAttention(
    (c_attn): Conv1D(nf=2304, nx=768)
    (c_proj): Conv1D(nf=768, nx=768)
    (attn_dropout): Dropout(p=0.1, inplace=False)
    (resid_dropout): Dropout(p=0.1, inplace=False)
  )
  (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (mlp): GPT2MLP(
    (c_fc): Conv1D(nf=3072, nx=768)
    (c_proj): Conv1D(nf=768, nx=3072)
    (act): NewGELUActivation()
    (dropout): Dropout(p=0.1, inplace=False)
  )
)

In [117]:
def gather_residual_activations(model, target_layer, inputs):
    target_act = None
    
    def gather_target_act_hook(mod, inputs):
        nonlocal target_act
        # Get residual stream from inputs
        target_act = inputs[0]
        return inputs
    
    # Hook into the attention block's input
    handle = model.transformer.h[target_layer].register_forward_pre_hook(gather_target_act_hook)
    
    model(inputs)
    handle.remove()
    return target_act

In [118]:
layer = 1
input_text = "The GreenBay Packers are"
tokenized_text = tokenize(input_text)
# embedding = tokenize_and_embed(input_text).to('cpu')
target_act = gather_residual_activations(model, layer, tokenized_text.to(device))


In [120]:
sae_acts = sae.encode(target_act.to('cpu'))
recon = sae.decode(sae_acts) # Reconstruction

In [121]:
1 - torch.mean((recon[:, 1:] - target_act[:, 1:].to(torch.float32).to('cpu')) **2) / (target_act[:, 1:].to(torch.float32).to('cpu').var())

tensor(0.9504, grad_fn=<RsubBackward1>)

In [113]:
(sae_acts > 1).sum(-1)

tensor([[29, 21, 24, 23, 25]])

In [124]:
values, inds = sae_acts.max(-1)
print(inds)
# tensor([[19418, 12725, 24128,   563, 24427,  9077, 13736, 16919, 22469,  4104]])

tensor([[18290, 13728, 20963, 20071,  7137]])


In [123]:
from IPython.display import IFrame
html_template = "https://neuronpedia.org/{}/{}/{}?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"

model_name = "gpt2-small"
sae_id = "1-res-jb" 

# 14561

def get_dashboard_html(sae_release = model_name, sae_id=sae_id, feature_idx=0):
    return html_template.format(sae_release, sae_id, feature_idx)

html = get_dashboard_html(feature_idx=20071)
IFrame(html, width=1200, height=600)

In [126]:
STEERING_COEFF = 10000
STEERING = True
def steering_hook(resid_pre, steering_vector, steering_coeff, steering_on=STEERING):
    """
    Apply steering vector to residual stream
    """
    # If output is a tuple, take the first element
    if isinstance(resid_pre, tuple):        
        resid_pre = resid_pre[0]        
        
    if resid_pre.shape[1] == 1:  # Skip if only one token        
        return
    
    position = resid_pre.shape[1]
    if steering_on:
        # Apply steering vector to all positions except the last one
        resid_pre[:, :position-1, :] += steering_coeff * steering_vector.to(device)

def hooked_generate(prompt, steering_vector, layer=layer, steering_coeff=STEERING_COEFF, seed=42, **kwargs):
    """
    Generate text with steering vector intervention
    """
    if seed is not None:
        torch.manual_seed(seed)
        
    def hook_fn(module, inputs):
        steering_hook(inputs[0], steering_vector, steering_coeff)
        return inputs
    
    # Register the hook
    hook = model.transformer.h[layer].register_forward_pre_hook(hook_fn)
    
    try:
        tokenized = tokenize(prompt).to(device)
        
        result = model.generate(
            tokenized,
            max_new_tokens=100,
            temperature=1.0,                    
            do_sample=True,
            pad_token_id=gpt2_tokenizer.eos_token_id,
            **kwargs
        )
        
        # Decode the generated text
        generated_text = gpt2_tokenizer.decode(result[0], skip_special_tokens=True)
        
    finally:
        # Always remove the hook to avoid memory leaks
        hook.remove()
        
    return generated_text

In [136]:
IDX = 20071
example_prompt = "Donald Trump was a"
steering_coeffs = list(np.linspace(5, 5000, 100))
steering_vector = sae.W_dec[IDX]
result = hooked_generate(example_prompt, steering_vector, steering_coeff=100)    
print(result)

Donald Trump was a second-round pick, Packers Packers 2 Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers Packers


In [133]:
perform_inference(example_prompt)

/Users/shayansadeghieh/Desktop/projects/mech_interp/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


'Hello.\n\nI\'m not sure if you\'re aware of the fact that the first time I saw this, I was in a bar with a bunch of guys and I was like, "Oh my God, this is so cool!" And I was like, "Oh my God, this is so cool!" And I was like, "Oh my God, this is so cool!" And I was like, "Oh my God, this is so cool!" And I was like, "Oh'